In [2]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_CSV_PATH = "ADNI_Seed_DementiaHKG.csv"

def build_seed_and_export_csv():
    print(f"==================================================")
    print(f"🕵️ 步骤 1: 加载 PrimeKG 数据库...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        all_nodes = set(df['x_name'].dropna()).union(set(df['y_name'].dropna()))
        print(f"   ✅ 成功加载 {RAW_KG_PATH}，共提取独立节点 {len(all_nodes)} 个。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print(f"\n==================================================")
    print(f"🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...")
    
    # 核心疾病与基因 (无对应的 ADNI 列，属于底层医学逻辑)
    core_seeds = {
        "Alzheimer disease": "Core_Disease", 
        "Dementia": "Core_Disease",
        "Cognitive impairment": "Core_Disease",
        "mild cognitive impairment": "Core_Disease", # 修正为小写
        "APOE": "Core_Gene",
        "MAPT": "Core_Gene",   
        "APP": "Core_Gene",    
        "PSEN1": "Core_Gene",
        "PSEN2": "Core_Gene",
        "TREM2": "Core_Gene"
    }
    print(f"   定义了 {len(core_seeds)} 个核心医学种子。")

    print(f"\n==================================================")
    print(f"📊 步骤 3: 补全 ADNI 临床与病史表型映射字典...")
    
    # 将 ADNI 特征映射到 PrimeKG 的标准 UMLS/MeSH 节点
    adni_mapping = {
        # --- NPI-Q 精神神经症状 ---
        "npiq_DEL": "Delusions",
        "npiq_HALL": "Hallucinations",
        "npiq_AGIT": "Agitation",
        "npiq_DEPD": "Depressivity",          # 根据检索结果修正
        "npiq_ANX": "Anxiety",
        "npiq_ELAT": "Euphoria",
        "npiq_APA": "Apathy",
        "npiq_DISN": "Disinhibition",
        "npiq_IRR": "Irritability",
        "npiq_MOT": "Restlessness",
        "npiq_NITE": "Sleep disturbance",     # 根据检索结果修正
        "npiq_APP": "Poor appetite",          # 根据检索结果修正

        # --- 认知与功能量表 (映射至底层神经表型) ---
        "mmse": "Cognitive impairment",
        "moca": "Cognitive impairment",
        "cdr": "Cognitive impairment",
        "cdrSum": "Cognitive impairment",
        "lm_imm": "Memory impairment",     
        "lm_del": "Memory impairment",
        "boston": "Aphasia",               
        "animal": "Aphasia",               
        "vege": "Aphasia",
        "trailA": "Impaired executive functioning", # 根据检索结果修正
        "trailB": "Impaired executive functioning", # 根据检索结果修正
        "digitB": "Short attention span",           # 根据检索结果修正
        "digitBL": "Short attention span",          # 根据检索结果修正
        "digitF": "Short attention span",           # 根据检索结果修正
        "digitFL": "Short attention span",          # 根据检索结果修正
        "faq_BILLS": "Impaired executive functioning", # 根据检索结果修正
        "faq_TAXES": "Impaired executive functioning", # 根据检索结果修正
        "faq_SHOPPING": "Memory impairment",
        "faq_REMDATES": "Memory impairment",
        "gds": "Depressivity",                      # 根据检索结果修正

        # --- EHR 既往病史映射 ---
        "his_CVHATT": "myocardial infarction", # 医学标准学名，全小写
        "his_CBSTROKE": "cerebral infarction", # 根据检索结果修正
        "his_HYPERTEN": "hypertension",        # 根据检索结果修正
        "his_PSYCDIS": "mental disorder",      # 根据检索结果修正
        "his_Alcohol": "Alcoholism",
        "his_DEPOTHR": "Depressivity"          # 根据检索结果修正
    }
    print(f"   定义了 {len(adni_mapping)} 个 ADNI 特征到 PrimeKG 节点的映射规则。")

    print(f"\n==================================================")
    print(f"🔍 步骤 4: 在 PrimeKG 中校验节点有效性...")
    
    valid_records = []
    
    # 校验核心种子
    print("   [校验核心种子]")
    for seed, category in core_seeds.items():
        if seed in all_nodes:
            valid_records.append({
                "ADNI_Feature": "Core_Prior_Knowledge",
                "PrimeKG_Node": seed,
                "Category": category
            })
            print(f"     ✅ 命中: {seed}")
        else:
            print(f"     ❌ 未命中: {seed} (将被丢弃)")

    # 校验 ADNI 映射
    print("   [校验 ADNI 映射特征]")
    for adni_feat, pkg_node in adni_mapping.items():
        if pkg_node in all_nodes:
            valid_records.append({
                "ADNI_Feature": adni_feat,
                "PrimeKG_Node": pkg_node,
                "Category": "Clinical_Mapping"
            })
            print(f"     ✅ 命中: {adni_feat} -> {pkg_node}")
        else:
            print(f"     ❌ 未命中: {adni_feat} -> {pkg_node} (请检查 PrimeKG 命名)")

    print(f"\n==================================================")
    print(f"🚫 步骤 5: 附加全局黑名单规则...")
    
    valid_records.append({"ADNI_Feature": "BLACKLIST_TYPE", "PrimeKG_Node": "drug", "Category": "Rule"})
    valid_records.append({"ADNI_Feature": "BLACKLIST_TYPE", "PrimeKG_Node": "exposure", "Category": "Rule"})
    valid_records.append({"ADNI_Feature": "BLACKLIST_REL", "PrimeKG_Node": "indication", "Category": "Rule"})
    valid_records.append({"ADNI_Feature": "BLACKLIST_REL", "PrimeKG_Node": "contraindication", "Category": "Rule"})
    valid_records.append({"ADNI_Feature": "BLACKLIST_REL", "PrimeKG_Node": "off-label use", "Category": "Rule"})
    print("   已将药物节点(drug)及用药关系等 5 条黑名单规则编入导出列表。")

    print(f"\n==================================================")
    print(f"💾 步骤 6: 导出为 CSV 文件...")
    
    out_df = pd.DataFrame(valid_records)
    out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
    
    print(f"   ✅ 导出完成！总计保留 {len(out_df)} 行有效配置。")
    print(f"   📄 文件已保存至: {OUTPUT_CSV_PATH}")
    print(f"==================================================")

if __name__ == "__main__":
    build_seed_and_export_csv()

🕵️ 步骤 1: 加载 PrimeKG 数据库...
   ✅ 成功加载 kg.csv，共提取独立节点 129262 个。

🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...
   定义了 10 个核心医学种子。

📊 步骤 3: 补全 ADNI 临床与病史表型映射字典...
   定义了 38 个 ADNI 特征到 PrimeKG 节点的映射规则。

🔍 步骤 4: 在 PrimeKG 中校验节点有效性...
   [校验核心种子]
     ✅ 命中: Alzheimer disease
     ✅ 命中: Dementia
     ✅ 命中: Cognitive impairment
     ❌ 未命中: mild cognitive impairment (将被丢弃)
     ✅ 命中: APOE
     ✅ 命中: MAPT
     ✅ 命中: APP
     ✅ 命中: PSEN1
     ✅ 命中: PSEN2
     ✅ 命中: TREM2
   [校验 ADNI 映射特征]
     ✅ 命中: npiq_DEL -> Delusions
     ✅ 命中: npiq_HALL -> Hallucinations
     ✅ 命中: npiq_AGIT -> Agitation
     ✅ 命中: npiq_DEPD -> Depressivity
     ✅ 命中: npiq_ANX -> Anxiety
     ✅ 命中: npiq_ELAT -> Euphoria
     ✅ 命中: npiq_APA -> Apathy
     ✅ 命中: npiq_DISN -> Disinhibition
     ✅ 命中: npiq_IRR -> Irritability
     ✅ 命中: npiq_MOT -> Restlessness
     ✅ 命中: npiq_NITE -> Sleep disturbance
     ✅ 命中: npiq_APP -> Poor appetite
     ✅ 命中: mmse -> Cognitive impairment
     ✅ 命中: moca -> Cognitive impairment
     ✅ 命中: cdr -> Cog